# Training in Practice: Epochs, Mini-Batches & Optimizers

CSCI 6379 · Topic 18. The practical knobs around gradient descent: epoch/batch/iteration, batch vs stochastic vs mini-batch GD, how many epochs, what batch size, and the optimizers (SGD, momentum, Adam) — each shown by experiment.

## Setup: a linear-regression dataset (N = 1,000)

True line y = 3x + 5 plus Gaussian noise. The noise floor puts the best possible MSE near 1.0.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
x = rng.uniform(-2, 2, 1000).astype(np.float32)
y = (3*x + 5 + rng.normal(0, 1, 1000)).astype(np.float32)
X = torch.tensor(x).unsqueeze(1); Y = torch.tensor(y).unsqueeze(1)
crit = nn.MSELoss()

## Experiment 1: batch vs SGD vs mini-batch

Same data, same learning rate — only how many samples feed each update changes. Watch the cost after ONE epoch.

In [ ]:
def run(mode, epochs, lr=0.05, bs=32, seed=0):
    torch.manual_seed(seed)
    m = nn.Linear(1, 1); nn.init.zeros_(m.weight); nn.init.zeros_(m.bias)
    opt = optim.SGD(m.parameters(), lr=lr)
    costs = []
    n = len(X)
    for ep in range(epochs):
        perm = torch.randperm(n)
        if mode == "batch":     idxs = [perm]                                  # 1 update/epoch
        elif mode == "sgd":     idxs = [perm[i:i+1] for i in range(n)]         # 1,000 updates/epoch
        else:                   idxs = [perm[i:i+bs] for i in range(0, n, bs)] # 32 updates/epoch
        for idx in idxs:
            opt.zero_grad(); loss = crit(m(X[idx]), Y[idx]); loss.backward(); opt.step()
            with torch.no_grad(): costs.append(crit(m(X), Y).item())
    return costs

cb = run("batch", 60); cs = run("sgd", 2); cm = run("minibatch", 2)
print("after ONE epoch:  batch %.2f (1 update)   mini-batch %.2f (32 updates)   SGD %.2f (1000 updates)"
      % (cb[0], cm[31], cs[999]))

plt.figure(figsize=(8,4))
plt.plot(cb, "o-", label="batch GD"); plt.plot(cm, label="mini-batch 32"); plt.plot(cs, lw=.7, label="SGD")
plt.xscale("log"); plt.yscale("log"); plt.xlabel("gradient updates"); plt.ylabel("cost (full set)")
plt.legend(); plt.grid(alpha=.3); plt.show()

## Experiment 2: how many epochs? what batch size?

Left idea: train/test accuracy climbs then plateaus — pick epochs by watching the curve. Right idea: smaller batches make more updates per epoch, so they reach the floor sooner.

In [ ]:
for bs in [1000, 128, 32, 8]:
    torch.manual_seed(0)
    m = nn.Linear(1,1); nn.init.zeros_(m.weight); nn.init.zeros_(m.bias)
    o = optim.SGD(m.parameters(), lr=0.05); hist=[]
    for ep in range(15):
        p = torch.randperm(len(X))
        for i in range(0, len(X), bs):
            idx = p[i:i+bs]
            o.zero_grad(); l = crit(m(X[idx]), Y[idx]); l.backward(); o.step()
        hist.append(crit(m(X), Y).item())
    lab = "full batch" if bs == 1000 else f"bs={bs}"
    print(f"{lab:11s} cost after epoch 1: {hist[0]:6.2f}   after 15: {hist[-1]:.3f}")
    plt.plot(range(1,16), hist, "o-", label=lab)
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("cost"); plt.legend(); plt.grid(alpha=.3); plt.show()

## Experiment 3: SGD vs momentum vs Adam

Same two-moons network, same data, same epochs — only the optimizer line changes.

In [ ]:
def make_moons(n=240, noise=0.15, seed=0):
    rg = np.random.default_rng(seed); k = n//2
    t = np.linspace(0, np.pi, k)
    Xm = np.vstack([np.c_[np.cos(t), np.sin(t)], np.c_[1-np.cos(t), 0.5-np.sin(t)]]) + rg.normal(0, noise, (2*k, 2))
    ym = np.r_[np.zeros(k), np.ones(k)]
    return torch.tensor(Xm, dtype=torch.float32), torch.tensor(ym, dtype=torch.float32).unsqueeze(1)
Xm, Ym = make_moons()

def run_opt(name, epochs=300):
    torch.manual_seed(0)
    m = nn.Sequential(nn.Linear(2,16), nn.ReLU(), nn.Linear(16,16), nn.ReLU(), nn.Linear(16,1))
    if name == "SGD":              opt = optim.SGD(m.parameters(), lr=0.05)
    elif name == "SGD + momentum": opt = optim.SGD(m.parameters(), lr=0.05, momentum=0.9)
    else:                          opt = optim.Adam(m.parameters(), lr=0.01)
    c = nn.BCEWithLogitsLoss(); hist=[]
    for _ in range(epochs):
        opt.zero_grad(); loss = c(m(Xm), Ym); loss.backward(); opt.step()
        hist.append(loss.item())
    acc = ((torch.sigmoid(m(Xm))>=0.5).float()==Ym).float().mean().item()
    return hist, acc

plt.figure(figsize=(8,4))
for name in ["SGD", "SGD + momentum", "Adam"]:
    h, acc = run_opt(name)
    print(f"{name:15s} final loss {h[-1]:.4f}   accuracy {acc*100:.1f}%")
    plt.plot(h, label=f"{name} ({acc*100:.0f}%)")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=.3); plt.show()

## Takeaways

- **mini-batch** = the everyday default (smooth enough, cheap enough, GPU-friendly); `DataLoader(batch_size=...)` implements it.
- pick **epochs** by watching train/test curves until they plateau.
- **Adam** is the "just works" optimizer default; momentum explains most of its edge over plain SGD.